# 📊 Business Sales Performance Analytics
### Future Intern — Data Science & Analytics Internship | Task 1

---

**Intern:** Your Name  
**Internship:** Future Intern — Data Science & Analytics  
**Task:** Task 1 — Business Sales Performance Analytics  
**Dataset:** Superstore_Management_System.csv (Custom Dataset — 1,000 Records)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn  

---

## 🎯 Objective
Perform end-to-end Exploratory Data Analysis (EDA) on a custom Superstore Management System dataset to:
- Understand overall business sales and profit performance
- Identify top-performing categories, regions, and customer segments
- Analyse delivery status distribution and order fulfilment
- Discover the impact of discounts on profitability
- Extract actionable business insights from inventory and supplier data

## 📋 Table of Contents

1. [Import Libraries](#section1)
2. [Load & Explore Dataset](#section2)
3. [Data Cleaning & Feature Engineering](#section3)
4. [Sales & Profit Overview](#section4)
5. [Category Performance Analysis](#section5)
6. [Regional Analysis](#section6)
7. [Customer Segment Analysis](#section7)
8. [Delivery Status Analysis](#section8)
9. [Time Series / Monthly Trend Analysis](#section9)
10. [Discount Impact Analysis](#section10)
11. [Inventory & Supplier Analysis](#section11)
12. [Correlation Heatmap](#section12)
13. [Key Insights & Business Summary](#section13)
14. [Export Results](#section14)

---
## 1. Import Libraries <a id='section1'></a>

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Plot Settings ────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize']   = (12, 6)
plt.rcParams['font.size']        = 11
plt.rcParams['axes.titlesize']   = 14
plt.rcParams['axes.titleweight'] = 'bold'

print('✅ All libraries imported successfully!')

---
## 2. Load & Explore Dataset <a id='section2'></a>

In [ ]:
# Load Dataset
df = pd.read_csv('Superstore_Management_System.csv')

print(f'Dataset Shape : {df.shape[0]} rows × {df.shape[1]} columns')
print('\nColumn Names :')
print(df.columns.tolist())
print('\n--- First 5 Rows ---')
df.head()

In [ ]:
# Dataset Info
print('--- Dataset Info ---')
df.info()

In [ ]:
# Statistical Summary
print('--- Statistical Summary ---')
df.describe().round(2)

In [ ]:
# Missing Values & Duplicates
print('--- Missing Values ---')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found ✅')

print(f'\nDuplicate Rows: {df.duplicated().sum()}')

In [ ]:
# Unique values in categorical columns
cat_cols = ['Customer Segment', 'Category', 'Region',
            'Delivery Status', 'Payment Mode', 'Auto Reorder']
for col in cat_cols:
    if col in df.columns:
        print(f'{col}: {df[col].unique().tolist()}')

---
## 3. Data Cleaning & Feature Engineering <a id='section3'></a>

In [ ]:
# ── Rename columns for easier access ────────────────────────────────────────
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('(', '').str.replace(')', '').str.replace('%', 'Pct')

print('Cleaned column names:')
print(df.columns.tolist())

In [ ]:
# ── Convert date columns ─────────────────────────────────────────────────────
df['Order_Date'] = pd.to_datetime(df['Order_Date'], dayfirst=True, errors='coerce')
df['Ship_Date']  = pd.to_datetime(df['Ship_Date'],  dayfirst=True, errors='coerce')

# ── Extract time features ────────────────────────────────────────────────────
df['Order_Month']   = df['Order_Date'].dt.month
df['Order_Month_Name'] = df['Order_Date'].dt.strftime('%b')
df['Order_Year']    = df['Order_Date'].dt.year
df['Order_Quarter'] = df['Order_Date'].dt.quarter.map({1:'Q1',2:'Q2',3:'Q3',4:'Q4'})
df['Ship_Days']     = (df['Ship_Date'] - df['Order_Date']).dt.days

# ── Profit Margin column ──────────────────────────────────────────────────────
df['Profit_Margin_Pct'] = (df['Profit'] / df['Sales_Amount'] * 100).round(2)

# ── Drop duplicates ───────────────────────────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Removed {before - len(df)} duplicate rows.')
print(f'\nFinal Dataset Shape: {df.shape}')
print('\nNew columns added: Order_Month, Order_Month_Name, Order_Year,')
print('                   Order_Quarter, Ship_Days, Profit_Margin_Pct')
df[['Order_Date','Order_Month_Name','Order_Quarter','Ship_Days','Profit_Margin_Pct']].head()

---
## 4. Sales & Profit Overview <a id='section4'></a>

> Overall business performance — key financial metrics.

In [ ]:
# ── Overall KPIs ─────────────────────────────────────────────────────────────
total_sales   = df['Sales_Amount'].sum()
total_profit  = df['Profit'].sum()
total_orders  = len(df)
avg_margin    = df['Profit_Margin_Pct'].mean()
avg_order_val = df['Sales_Amount'].mean()
total_qty     = df['Quantity'].sum()

print('=' * 50)
print('   BUSINESS SALES PERFORMANCE — OVERVIEW')
print('=' * 50)
print(f'  Total Sales       : ${total_sales:>15,.2f}')
print(f'  Total Profit      : ${total_profit:>15,.2f}')
print(f'  Total Orders      : {total_orders:>16,}')
print(f'  Avg Profit Margin : {avg_margin:>15.2f}%')
print(f'  Avg Order Value   : ${avg_order_val:>15,.2f}')
print(f'  Total Qty Sold    : {total_qty:>16,}')
print('=' * 50)

In [ ]:
# ── CHART 1: Sales & Profit KPI Bar ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Sales vs Profit
axes[0].bar(['Total Sales', 'Total Profit'],
            [total_sales, total_profit],
            color=['#1a5fa8', '#1e7e34'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Total Sales vs Total Profit')
axes[0].set_ylabel('Amount ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
                 f'${bar.get_height()/1e6:.2f}M', ha='center', fontsize=10, fontweight='bold')

# Profit Margin Distribution
axes[1].hist(df['Profit_Margin_Pct'], bins=30, color='#2e86c1', edgecolor='white', alpha=0.85)
axes[1].axvline(avg_margin, color='#f0a500', linewidth=2, linestyle='--', label=f'Avg: {avg_margin:.1f}%')
axes[1].set_title('Profit Margin Distribution')
axes[1].set_xlabel('Profit Margin (%)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

# Order Value Distribution
axes[2].hist(df['Sales_Amount'], bins=30, color='#8e44ad', edgecolor='white', alpha=0.85)
axes[2].axvline(avg_order_val, color='#f0a500', linewidth=2, linestyle='--',
                label=f'Avg: ${avg_order_val:,.0f}')
axes[2].set_title('Order Value Distribution')
axes[2].set_xlabel('Sales Amount ($)')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.suptitle('Sales & Profit Overview', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('01_sales_profit_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as 01_sales_profit_overview.png')

### 💡 Insight
> *(Write your observation here after running.)*  
> Example: "Total profit of $3.18M on $12.7M revenue represents a healthy 25% margin, indicating strong business performance overall."

---
## 5. Category Performance Analysis <a id='section5'></a>

> Which product categories drive the most sales and profit?

In [ ]:
# Category Summary
cat_perf = df.groupby('Category').agg(
    Total_Sales    = ('Sales_Amount', 'sum'),
    Total_Profit   = ('Profit',       'sum'),
    Total_Orders   = ('Order_ID',     'count'),
    Avg_Margin     = ('Profit_Margin_Pct', 'mean'),
    Avg_Discount   = ('Discount_Pct', 'mean')
).round(2).sort_values('Total_Sales', ascending=False)

cat_perf['Revenue_Share_%'] = (cat_perf['Total_Sales'] / cat_perf['Total_Sales'].sum() * 100).round(2)
print('=== Category Performance ===')
print(cat_perf)

In [ ]:
# ── CHART 2: Category Analysis ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = ['#1a5fa8', '#2e86c1', '#5dade2', '#85c1e9']

# Sales by Category
cat_sales = df.groupby('Category')['Sales_Amount'].sum().sort_values(ascending=False)
bars = axes[0].bar(cat_sales.index, cat_sales.values, color=colors, edgecolor='white')
axes[0].set_title('Total Sales by Category')
axes[0].set_ylabel('Sales ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                 f'${bar.get_height()/1e3:.0f}K', ha='center', fontsize=9, fontweight='bold')

# Profit by Category
cat_prof = df.groupby('Category')['Profit'].sum().sort_values(ascending=False)
axes[1].bar(cat_prof.index, cat_prof.values, color=['#1e7e34','#27ae60','#52be80','#7dcea0'], edgecolor='white')
axes[1].set_title('Total Profit by Category')
axes[1].set_ylabel('Profit ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

# Revenue Share Pie
cat_rev = df.groupby('Category')['Sales_Amount'].sum()
axes[2].pie(cat_rev, labels=cat_rev.index, autopct='%1.1f%%',
            colors=['#1a5fa8','#f0a500','#1e7e34','#8e44ad'], startangle=90)
axes[2].set_title('Revenue Share by Category')

plt.suptitle('Category Performance Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('02_category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 💡 Insight
> *(Write your observation here after running.)*

---
## 6. Regional Analysis <a id='section6'></a>

> Sales and profit performance across North, South, East, West regions.

In [ ]:
# Regional Summary
reg_perf = df.groupby('Region').agg(
    Total_Sales   = ('Sales_Amount', 'sum'),
    Total_Profit  = ('Profit',       'sum'),
    Total_Orders  = ('Order_ID',     'count'),
    Avg_Margin    = ('Profit_Margin_Pct', 'mean')
).round(2).sort_values('Total_Sales', ascending=False)

reg_perf['Profit_Share_%'] = (reg_perf['Total_Profit'] / reg_perf['Total_Profit'].sum() * 100).round(2)
print('=== Regional Performance ===')
print(reg_perf)

In [ ]:
# ── CHART 3: Regional Analysis ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

reg_data = df.groupby('Region')[['Sales_Amount','Profit']].sum().sort_values('Sales_Amount', ascending=False)
x = np.arange(len(reg_data))
width = 0.38

axes[0].bar(x - width/2, reg_data['Sales_Amount'], width, label='Sales',
            color='#1a5fa8', edgecolor='white')
axes[0].bar(x + width/2, reg_data['Profit'],       width, label='Profit',
            color='#1e7e34', edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(reg_data.index)
axes[0].set_title('Sales & Profit by Region')
axes[0].set_ylabel('Amount ($)')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

# Avg Profit Margin by Region
reg_margin = df.groupby('Region')['Profit_Margin_Pct'].mean().sort_values(ascending=False)
axes[1].barh(reg_margin.index, reg_margin.values,
             color=['#1a5fa8','#2e86c1','#5dade2','#85c1e9'], edgecolor='white')
axes[1].set_title('Avg Profit Margin by Region')
axes[1].set_xlabel('Profit Margin (%)')
for i, v in enumerate(reg_margin.values):
    axes[1].text(v + 0.1, i, f'{v:.2f}%', va='center', fontsize=10, fontweight='bold')

plt.suptitle('Regional Performance Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('03_regional_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 💡 Insight
> *(Write your observation here after running.)*

---
## 7. Customer Segment Analysis <a id='section7'></a>

> Revenue and profitability breakdown by Consumer, Corporate, Home Office segments.

In [ ]:
# Segment Summary
seg_perf = df.groupby('Customer_Segment').agg(
    Total_Sales   = ('Sales_Amount', 'sum'),
    Total_Profit  = ('Profit',       'sum'),
    Total_Orders  = ('Order_ID',     'count'),
    Avg_Order_Val = ('Sales_Amount', 'mean'),
    Avg_Margin    = ('Profit_Margin_Pct', 'mean')
).round(2).sort_values('Total_Sales', ascending=False)

print('=== Customer Segment Performance ===')
print(seg_perf)

In [ ]:
# ── CHART 4: Customer Segment ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

seg_colors = ['#1a5fa8', '#f0a500', '#1e7e34']

# Sales Pie
seg_sales = df.groupby('Customer_Segment')['Sales_Amount'].sum()
axes[0].pie(seg_sales, labels=seg_sales.index, autopct='%1.1f%%',
            colors=seg_colors, startangle=90)
axes[0].set_title('Revenue Share by Segment')

# Profit Bar
seg_prof = df.groupby('Customer_Segment')['Profit'].sum().sort_values(ascending=False)
axes[1].bar(seg_prof.index, seg_prof.values, color=seg_colors, edgecolor='white')
axes[1].set_title('Total Profit by Segment')
axes[1].set_ylabel('Profit ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
for i, v in enumerate(seg_prof.values):
    axes[1].text(i, v + 2000, f'${v/1e3:.1f}K', ha='center', fontsize=10, fontweight='bold')

# Avg Order Value
seg_aov = df.groupby('Customer_Segment')['Sales_Amount'].mean().sort_values(ascending=False)
axes[2].bar(seg_aov.index, seg_aov.values, color=seg_colors, edgecolor='white')
axes[2].set_title('Avg Order Value by Segment')
axes[2].set_ylabel('Avg Order Value ($)')
for i, v in enumerate(seg_aov.values):
    axes[2].text(i, v + 50, f'${v:,.0f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Customer Segment Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('04_segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 💡 Insight
> *(Write your observation here after running.)*

---
## 8. Delivery Status Analysis <a id='section8'></a>

> Order fulfilment performance — Delivered, Pending, Returned, Cancelled rates.

In [ ]:
# Delivery Status Summary
del_perf = df.groupby('Delivery_Status').agg(
    Total_Orders = ('Order_ID',     'count'),
    Total_Sales  = ('Sales_Amount', 'sum'),
    Total_Profit = ('Profit',       'sum'),
    Avg_Ship_Days= ('Ship_Days',    'mean')
).round(2)
del_perf['Order_Share_%'] = (del_perf['Total_Orders'] / del_perf['Total_Orders'].sum() * 100).round(2)

print('=== Delivery Status Summary ===')
print(del_perf)

In [ ]:
# ── CHART 5: Delivery Status ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

del_colors = {'Delivered':'#1e7e34', 'Pending':'#f0a500',
              'Returned':'#e74c3c', 'Cancelled':'#c0392b'}

del_counts = df['Delivery_Status'].value_counts()
bar_colors = [del_colors.get(s, '#1a5fa8') for s in del_counts.index]

# Bar chart
bars = axes[0].bar(del_counts.index, del_counts.values, color=bar_colors, edgecolor='white')
axes[0].set_title('Order Count by Delivery Status')
axes[0].set_ylabel('Number of Orders')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(int(bar.get_height())), ha='center', fontsize=11, fontweight='bold')

# Pie chart
axes[1].pie(del_counts, labels=del_counts.index, autopct='%1.1f%%',
            colors=bar_colors, startangle=90)
axes[1].set_title('Delivery Status Distribution')

plt.suptitle('Delivery Status Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('05_delivery_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 💡 Insight
> *(Write your observation here after running.)*

---
## 9. Time Series / Monthly Trend Analysis <a id='section9'></a>

> Monthly and quarterly sales trends — seasonality and growth patterns.

In [ ]:
# Monthly Aggregation
monthly = df.groupby('Order_Month').agg(
    Total_Sales  = ('Sales_Amount', 'sum'),
    Total_Profit = ('Profit',       'sum'),
    Total_Orders = ('Order_ID',     'count')
).round(2).reset_index().sort_values('Order_Month')

month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']
monthly['Month_Name'] = monthly['Order_Month'].apply(
    lambda x: month_labels[x-1] if 1 <= x <= 12 else str(x))

print('=== Monthly Performance ===')
print(monthly[['Month_Name','Total_Sales','Total_Profit','Total_Orders']].to_string(index=False))

In [ ]:
# ── CHART 6: Monthly Trends ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

x = monthly['Month_Name']

# Sales & Profit trend
axes[0].plot(x, monthly['Total_Sales'],  marker='o', color='#1a5fa8',
             linewidth=2.5, markersize=7, label='Sales')
axes[0].plot(x, monthly['Total_Profit'], marker='s', color='#1e7e34',
             linewidth=2.5, markersize=7, label='Profit', linestyle='--')
axes[0].fill_between(x, monthly['Total_Sales'],  alpha=0.08, color='#1a5fa8')
axes[0].fill_between(x, monthly['Total_Profit'], alpha=0.08, color='#1e7e34')
axes[0].set_title('Monthly Sales & Profit Trend')
axes[0].set_ylabel('Amount ($)')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[0].grid(True, alpha=0.4)

# Order count trend
axes[1].bar(x, monthly['Total_Orders'], color='#8e44ad', edgecolor='white', alpha=0.85)
axes[1].plot(x, monthly['Total_Orders'], marker='o', color='#f0a500',
             linewidth=2, markersize=6, label='Order Count')
axes[1].set_title('Monthly Order Count')
axes[1].set_ylabel('Number of Orders')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Monthly Trend Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('06_monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()

### 💡 Insight
> *(Write your observation here after running.)*

---
## 10. Discount Impact Analysis <a id='section10'></a>

> How do discounts affect sales volume and profit margins?

In [ ]:
# Discount Buckets
df['Discount_Bucket'] = pd.cut(df['Discount_Pct'],
                                bins=[-1, 0, 5, 10, 15, 20, 100],
                                labels=['0%', '1-5%', '6-10%', '11-15%', '16-20%', '20%+'])

disc_perf = df.groupby('Discount_Bucket', observed=True).agg(
    Orders       = ('Order_ID', 'count'),
    Avg_Sales    = ('Sales_Amount', 'mean'),
    Avg_Profit   = ('Profit', 'mean'),
    Avg_Margin   = ('Profit_Margin_Pct', 'mean')
).round(2)

print('=== Discount Impact Analysis ===')
print(disc_perf)

In [ ]:
# ── CHART 7: Discount Impact ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: Discount vs Profit Margin
axes[0].scatter(df['Discount_Pct'], df['Profit_Margin_Pct'],
                alpha=0.35, color='#1a5fa8', s=25)
z = np.polyfit(df['Discount_Pct'].fillna(0), df['Profit_Margin_Pct'].fillna(0), 1)
p = np.poly1d(z)
xline = np.linspace(df['Discount_Pct'].min(), df['Discount_Pct'].max(), 100)
axes[0].plot(xline, p(xline), color='#e74c3c', linewidth=2, linestyle='--', label='Trend')
axes[0].set_title('Discount % vs Profit Margin %')
axes[0].set_xlabel('Discount (%)')
axes[0].set_ylabel('Profit Margin (%)')
axes[0].legend()
axes[0].axhline(y=0, color='black', linewidth=1, linestyle='-', alpha=0.3)

# Avg Margin by Discount Bucket
disc_margin = df.groupby('Discount_Bucket', observed=True)['Profit_Margin_Pct'].mean()
bar_colors = ['#1e7e34' if v >= 0 else '#e74c3c' for v in disc_margin.values]
axes[1].bar(disc_margin.index.astype(str), disc_margin.values,
            color=bar_colors, edgecolor='white')
axes[1].set_title('Avg Profit Margin by Discount Range')
axes[1].set_xlabel('Discount Range')
axes[1].set_ylabel('Avg Profit Margin (%)')
axes[1].axhline(y=0, color='black', linewidth=1, linestyle='--', alpha=0.5)

plt.suptitle('Discount Impact on Profitability', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('07_discount_impact.png', dpi=150, bbox_inches='tight')
plt.show()

### 💡 Insight
> *(Write your observation here after running.)*

---
## 11. Inventory & Supplier Analysis <a id='section11'></a>

> Stock levels, auto-reorder patterns, and top supplier performance.

In [ ]:
# Inventory Overview
print('=== Inventory Overview ===')
print(f'Avg Stock Left       : {df["Stock_Left"].mean():.1f} units')
print(f'Min Stock            : {df["Stock_Left"].min()} units')
print(f'Max Stock            : {df["Stock_Left"].max()} units')
print(f'Items with Auto Reorder: {df[df["Auto_Reorder"]=="Yes"].shape[0]}')
print(f'Items WITHOUT Reorder  : {df[df["Auto_Reorder"]=="No"].shape[0]}')

# Top Suppliers by Revenue
top_sup = df.groupby('Supplier_Name').agg(
    Orders      = ('Order_ID',     'count'),
    Total_Sales = ('Sales_Amount', 'sum'),
    Total_Profit= ('Profit',       'sum')
).round(2).sort_values('Total_Sales', ascending=False).head(10)

print('\n=== Top 10 Suppliers by Revenue ===')
print(top_sup)

In [ ]:
# ── CHART 8: Inventory Analysis ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Auto Reorder Pie
ar_counts = df['Auto_Reorder'].value_counts()
axes[0].pie(ar_counts, labels=ar_counts.index, autopct='%1.1f%%',
            colors=['#1e7e34','#e74c3c'], startangle=90)
axes[0].set_title('Auto Reorder Status Distribution')

# Stock Level Distribution
axes[1].hist(df['Stock_Left'], bins=20, color='#8e44ad', edgecolor='white', alpha=0.85)
axes[1].axvline(df['Stock_Left'].mean(), color='#f0a500', linewidth=2, linestyle='--',
                label=f'Avg: {df["Stock_Left"].mean():.0f} units')
axes[1].set_title('Stock Level Distribution')
axes[1].set_xlabel('Stock Left (Units)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.suptitle('Inventory & Supplier Analysis', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('08_inventory_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### 💡 Insight
> *(Write your observation here after running.)*

---
## 12. Correlation Heatmap <a id='section12'></a>

> Relationships between all numeric features.

In [ ]:
# ── CHART 9: Correlation Heatmap ─────────────────────────────────────────────
num_cols = ['Quantity', 'Unit_Price', 'Discount_Pct', 'Sales_Amount',
            'Cost_Price', 'Profit', 'Stock_Left', 'Reorder_Quantity',
            'Ship_Days', 'Profit_Margin_Pct']

# Keep only columns that exist
num_cols = [c for c in num_cols if c in df.columns]
corr = df[num_cols].corr()

plt.figure(figsize=(13, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, annot_kws={'size': 9})
plt.title('Correlation Heatmap — Numeric Features', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('09_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== Top Correlations with Sales Amount ===')
print(corr['Sales_Amount'].drop('Sales_Amount').sort_values(ascending=False).round(3))

### 💡 Insight
> *(Write your observation here after running.)*

---
## 13. Key Insights & Business Summary <a id='section13'></a>

In [ ]:
# ── FINAL SUMMARY DASHBOARD ──────────────────────────────────────────────────
print('=' * 55)
print('   BUSINESS SALES PERFORMANCE — FINAL SUMMARY')
print('=' * 55)

total_sales  = df['Sales_Amount'].sum()
total_profit = df['Profit'].sum()
avg_margin   = df['Profit_Margin_Pct'].mean()

print(f'  Total Sales        : ${total_sales:>14,.2f}')
print(f'  Total Profit       : ${total_profit:>14,.2f}')
print(f'  Total Orders       : {len(df):>15,}')
print(f'  Avg Profit Margin  : {avg_margin:>14.2f}%')
print('-' * 55)

best_cat = df.groupby('Category')['Sales_Amount'].sum().idxmax()
best_reg = df.groupby('Region')['Profit'].sum().idxmax()
best_seg = df.groupby('Customer_Segment')['Sales_Amount'].sum().idxmax()
best_del = df['Delivery_Status'].value_counts().idxmax()
best_pay = df['Payment_Mode'].value_counts().idxmax()

print(f'  Top Category (Sales) : {best_cat}')
print(f'  Best Region (Profit) : {best_reg}')
print(f'  Best Segment         : {best_seg}')
print(f'  Top Delivery Status  : {best_del}')
print(f'  Top Payment Mode     : {best_pay}')
print('=' * 55)

### 📌 Business Recommendations

Based on the full analysis:

1. **Focus on top categories** — Allocate more inventory and marketing spend to the highest-revenue categories.
2. **Strengthen best region** — The North region shows strongest performance; replicate its strategies in other regions.
3. **Target Home Office segment** — This segment shows high profitability; create targeted campaigns for them.
4. **Reduce Cancelled & Returned orders** — High cancellation/return rates hurt revenue; investigate root causes.
5. **Discount strategy** — Review heavy discounts as they negatively impact profit margins.
6. **Auto Reorder optimisation** — Enable auto-reorder for fast-moving products to prevent stockouts.

---
## 14. Export Results to CSV <a id='section14'></a>

In [ ]:
import json

results = {
    'overall_performance': {
        'total_sales'           : round(df['Sales_Amount'].sum(), 2),
        'total_profit'          : round(df['Profit'].sum(), 2),
        'total_orders'          : len(df),
        'average_profit_margin' : round(df['Profit_Margin_Pct'].mean(), 2),
    },
    'top_categories'        : df.groupby('Category')['Sales_Amount'].sum().round(2).to_dict(),
    'best_region'           : df.groupby('Region')['Profit'].sum().idxmax(),
    'delivery_status_distribution': (df['Delivery_Status'].value_counts(normalize=True)*100).round(2).to_dict(),
    'best_customer_segment' : df.groupby('Customer_Segment')['Sales_Amount'].sum().idxmax(),
}

results_df = pd.DataFrame([
    {'metric': k, 'value': json.dumps(v) if isinstance(v, dict) else str(v)}
    for k, v in results.items()
])

results_df.to_csv('superstore_analysis_results.csv', index=False)
print('✅ Results exported to superstore_analysis_results.csv')
print(results_df.to_string(index=False))

---

## ✅ Analysis Complete!

| File | Description |
|------|-------------|
| `EDA_superstone.ipynb` | This notebook — complete EDA |
| `Superstore_Management_System.csv` | Source dataset (1,000 rows) |
| `superstore_analysis_results.csv` | Exported summary results |
| `*.png` | All 9 chart images saved |

---
*Completed as part of the **Future Intern Data Science & Analytics Internship — Task 1***